# M3 step 10: does the stamp work on GenImage photos? (task 6)

M3 was trained on COCO photos, but RQ3 stamps GenImage photos. AI generated images are often smoother,
which can make the stamp harder to hide and to read. This check needs only the GenImage test split,
not the detectors.

A few hundred Real and a few hundred AI photos are drawn from the detectors' test split (seed 42),
stamped, saved as PNG and as JPEG quality 90, and read back. Bit accuracy, valid verdict and PSNR are
reported separately for Real and AI photos, and per generator.

If bits after PNG fall below 95% for either class, tell Rishitha: a short fine-tune may be needed,
which would go in a new notebook and a new run folder.

**GPU:** not needed. About 5 to 10 minutes on a CPU for 600 photos.

## Setup

**Kaggle:** turn on *Settings > Accelerator > GPU T4 x2* (only notebooks 8, 9 and 11 need it) and
*Internet on*, and attach the `veristamp-m3-inputs` dataset (README.md, "Running on Kaggle"). The first
cell then finds the code, installs what is missing and moves into `m3_experiments`. No path to edit.

**Colab or a local machine:** set the paths once in `settings.py`, then run from the `m3_experiments`
folder.

In [ ]:
# Kaggle setup: find the dataset wherever Kaggle put it, copy the code, point the notebooks at the files.
import os, shutil, subprocess, sys
from pathlib import Path

def find(rel, max_depth=6):
    for depth in range(max_depth + 1):
        hits = sorted(Path('/kaggle/input').glob('/'.join(['*'] * depth + [rel])))
        if hits:
            return hits[0]
    return None

if Path('/kaggle/input').is_dir() and not Path('settings.py').exists():
    code = find('m3_experiments/settings.py')
    model = find('m3/best.pt')
    print('Code found at: ', code)
    print('Model found at:', model)
    if code is None or model is None:
        print(subprocess.run(['find', '/kaggle/input', '-maxdepth', '4'], capture_output=True, text=True).stdout[:3000])
        raise SystemExit('The veristamp-m3-inputs dataset is not attached, or has a different layout (see the list above).')
    repo = Path('/kaggle/working/VeriStamp')
    if not (repo / 'm3_experiments' / 'settings.py').exists():
        shutil.rmtree(repo, ignore_errors=True)          # removes a failed earlier clone
        shutil.copytree(code.parent.parent, repo, ignore=shutil.ignore_patterns('.git'))
    # Each item is found on its own, so the inputs can come in several zips or datasets.
    paths = {'M3E_MODEL_PATH': 'm3/best.pt', 'M3E_TEST_DIR': 'm3/test', 'M3E_TEST_MANIFEST': 'm3/test_manifest.csv',
             'M3E_GENIMAGE_SPLIT_DIR': 'genimage/splits', 'M3E_GENIMAGE_ROOT': 'genimage/images',
             'M3E_M1_CHECKPOINT': 'm1/best.pt', 'M3E_M1_VSLIB_DIR': 'm1/vslib_m1_final'}
    for name, rel in paths.items():
        hit = find(rel)
        os.environ[name] = str(hit if hit else model.parent.parent / rel)   # the notebooks run from here inherit these
        print(f'  {rel:<22} {"found" if hit else "not in the inputs"}')
    os.environ['M3E_WORK_DIR'] = '/kaggle/working/m3_work'
    os.chdir(repo / 'm3_experiments')
    for pkg in ('diffusers', 'lpips'):
        try:
            __import__(pkg)
        except ImportError:
            subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', pkg], check=True)
print('Working in', Path.cwd())

In [ ]:
# On Colab instead, uncomment and run once per session:
# from google.colab import drive; drive.mount('/content/drive')
# !git clone https://github.com/KruthikaVirupakshappa/VeriStamp.git /content/VeriStamp
# %cd /content/VeriStamp/m3_experiments
# !pip install -q diffusers transformers accelerate scikit-image lpips

import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))
import numpy as np
import pandas as pd
import torch
from PIL import Image
import matplotlib.pyplot as plt

import settings as S
import m3_eval as E
mw = S.setup()                      # imports ../m3_watermark/m3_watermark.py as it is
torch.manual_seed(S.SEED)
np.random.seed(S.SEED)

m3 = mw.load_m3(S.MODEL_PATH)       # load once, reuse for every photo
DEVICE = m3.device
print('Model epoch', m3.epoch, 'on', DEVICE)

## Sample the GenImage test split

In [ ]:
N_PER_CLASS = 300
sample = E.load_genimage_test(S.GENIMAGE_SPLIT_DIR, S.GENIMAGE_ROOT, N_PER_CLASS, S.SEED)
sample['class'] = sample['label'].map({0: 'Real', 1: 'AI'})
print(sample.groupby(['class', 'generator']).size())

## Stamp, save, read back

In [ ]:
EDITS = {'Saved as PNG': lambda im: E.save_as(im, 'PNG'), 'JPEG quality 90': lambda im: E.save_as(im, 'JPEG', 90)}
rows = []
for _, s in E.tqdm(sample.iterrows(), total=len(sample)):
    photo = E.open_rgb(s['path'])
    p = E.payload_for(str(s['image_id']), S.SEED)
    expected = ''.join(map(str, mw.encode_payload(p['verdict'], p['p_ai'], image_tag=p['image_tag'])))
    stamped = E.save_as(mw.embed(m3, photo, p['verdict'], p['p_ai'], image_tag=p['image_tag']), 'PNG')
    quality = {'psnr': E.psnr(stamped, photo), 'ssim': E.ssim(stamped, photo)}
    unstamped_valid = mw.extract(m3, E.save_as(photo, 'PNG'))['valid']
    for name, edit in EDITS.items():
        res = E.score_read(mw, m3, edit(stamped), expected, p['verdict'])
        rows.append({'image_id': s['image_id'], 'class': s['class'], 'generator': s['generator'],
                     'edit': name, 'width': photo.width, 'height': photo.height, **quality,
                     'unstamped_read_valid': unstamped_valid, **res})
per_photo = pd.DataFrame(rows)

## Results by class

In [ ]:
by_class = E.summarise(per_photo, by=('class', 'edit'))
psnr_table = per_photo[per_photo.edit == 'Saved as PNG'].groupby('class')[['psnr', 'ssim']].mean()
by_class = by_class.merge(psnr_table, left_on='class', right_index=True)
by_generator = E.summarise(per_photo, by=('generator', 'class', 'edit'))
false_valid = per_photo[per_photo.edit == 'Saved as PNG'].groupby('class')['unstamped_read_valid'].sum()
display(E.as_percent(by_class))
display(E.as_percent(by_generator))
print('Unstamped photos read as valid:', false_valid.to_dict())
print('COCO reference (notebook 4, PNG): 99.85% bits, 96.0% valid verdict, 31.50 dB')

In [ ]:
png = by_class[by_class.edit == 'Saved as PNG']
low = png[png.bit_accuracy < 0.95]
if len(low):
    print('BELOW 95% BITS AFTER PNG for:', list(low['class']), '-> tell Rishitha before RQ3.')
else:
    print('Both classes are above 95% bits after PNG: the COCO-trained model can be used for RQ3.')

## Save

In [ ]:
E.save_results(S.RESULTS_DIR, 'genimage_check', per_photo=per_photo, summary=by_class,
               meta={'n_per_class': N_PER_CLASS, 'split_dir': str(S.GENIMAGE_SPLIT_DIR), 'seed': S.SEED,
                     'false_valid_unstamped': {k: int(v) for k, v in false_valid.items()},
                     'environment': E.environment_info()})
by_generator.to_csv(S.RESULTS_DIR / 'genimage_check_by_generator.csv', index=False)
print(E.to_markdown(E.as_percent(by_class)))